# Planning limits: horizon, feedback, and subgoals

**Claim status: illustrative analogue.** This lesson compares planners in a small state-space control task. It does not reproduce the paper’s Meta-World results or test a learned visual world model.

## Learning outcome

Compare a short imagined rollout with feedback and nearby targets. See how goal distance and rollout horizon affect a simple planner.

## What this lesson uses

- Environment: Gymnasium `MountainCarContinuous-v0`.
- Model: the exact, known state transition from Gymnasium.
- Methods: pure imagination, feedback MPC, subgoal MPC using a scripted momentum reference, and a momentum baseline.
- Run: 20 paired reset seeds, goals at 0.0, 0.25, and 0.45, rollout horizons 5 and 20, and at most 300 steps per episode. Success means reaching the goal position with non-negative velocity.
- Tools: Python, NumPy, Matplotlib, Gymnasium, and Jupyter. No download or robot is needed.

The exact model keeps prediction error out of this comparison. It also means this lesson cannot measure the value of feedback when a model is wrong or the environment changes. Subgoal targets come from a same-seed rollout of the simple momentum rule. This is a scripted teaching aid, not an expert demonstration.


## Paper-reported findings

Version 1 of [The Planning Limits of Latent World Models](https://arxiv.org/html/2609.39235v1) reports results on 16 Meta-World tasks, with 20 paired episodes per task. Its V-JEPA 2.1 planner reports 23% success for pure imagination with a 5-step rollout and 30% for 5-step feedback MPC. A 30-step feedback MPC reports 52%, and 5-step MPC with nearby expert subgoals reports 76%. These are paper results. They are not measurements from this notebook.

The paper describes pure imagination as planning from its own predicted states after the first observation. MPC executes one action, observes the result, and replans. The paper’s subgoals come from an expert rollout of the same episode. This lesson uses a five-step-ahead target from a same-seed momentum-controller reference path instead.


In [ ]:
import math
import platform
import time
from collections import defaultdict

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

GOALS = (0.0, 0.25, 0.45)
HORIZONS = (5, 20)
SEEDS = tuple(range(1000, 1020))
MAX_STEPS = 300
CEM_CANDIDATES = 64
CEM_ELITES = 8
CEM_ITERATIONS = 2

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor() or platform.machine()}")
print(f"Logical CPU count: {__import__('os').cpu_count()}")
print(f"Seeds: {SEEDS[0]}–{SEEDS[-1]} ({len(SEEDS)} paired starts)")
print(f"CEM: {CEM_CANDIDATES} candidates, {CEM_ELITES} elites, {CEM_ITERATIONS} iterations")
print(f"Episode limit: {MAX_STEPS} steps")

## The exact transition model

The state is position and velocity. The action is a force from -1 to 1. The helper below follows Gymnasium’s published transition and supports a batch of candidate action sequences. The next cell checks it against the installed environment before any planner uses it.

In [ ]:
MIN_POSITION = -1.2
MAX_POSITION = 0.6
MAX_SPEED = 0.07
POWER = 0.0015
GRAVITY = 0.0025

def model_step(states, actions):
    """Apply one exact continuous MountainCar transition to one or many states."""
    states = np.asarray(states, dtype=np.float32)
    actions = np.asarray(actions, dtype=np.float32)
    position = states[..., 0]
    velocity = states[..., 1]
    force = np.clip(actions, -1.0, 1.0)
    next_velocity = np.clip(velocity + force * POWER - GRAVITY * np.cos(3 * position), -MAX_SPEED, MAX_SPEED)
    next_position = np.clip(position + next_velocity, MIN_POSITION, MAX_POSITION)
    next_velocity = np.where((next_position == MIN_POSITION) & (next_velocity < 0), 0.0, next_velocity)
    return np.stack((next_position, next_velocity), axis=-1).astype(np.float32)

In [ ]:
# Check the local model at ordinary states and both position limits.
transition_cases = [
    ([-0.55, 0.0], 1.0),
    ([-0.50, 0.04], -0.5),
    ([-1.20, -0.03], -1.0),
    ([0.59, 0.06], 1.0),
]
check_env = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_STEPS)
check_env.reset(seed=1)
transition_errors = []
for state, action in transition_cases:
    check_env.unwrapped.state = np.asarray(state, dtype=np.float32)
    expected, _, _, _, _ = check_env.unwrapped.step(np.asarray([action], dtype=np.float32))
    actual = model_step(np.asarray(state, dtype=np.float32), action)
    transition_errors.append(float(np.max(np.abs(expected - actual))))
check_env.close()
max_transition_error = max(transition_errors)
print("Maximum absolute transition error:", max_transition_error)
assert max_transition_error <= 1e-6, "Local transition does not match this Gymnasium install"
print("Transition check passed at tolerance 1e-6.")

## Planner and baselines

CEM samples action sequences, scores their predicted states, and keeps the best candidates. Each planner condition uses the same candidate count, elite count, and iteration count. The score is the closest predicted state to the active position target, with a small action cost.

- **Pure imagination** uses its first observation, then updates its internal state with its own predictions. It does not read later observations when it chooses actions.
- **Feedback MPC** uses the latest environment state and replans after every action.
- **Subgoal MPC** uses feedback MPC with a target five steps ahead on a same-seed path from the momentum baseline. It uses the same model and CEM budget.
- **Momentum baseline** applies force +1 when velocity is zero or positive, and -1 when velocity is negative. It does not use the horizon. Its path is also the scripted reference for subgoal MPC.


In [ ]:
def cem_plan(start_state, target, horizon, rng):
    """Return the best action sequence found under the fixed CEM budget."""
    mean = np.zeros(horizon, dtype=np.float32)
    std = np.full(horizon, 0.7, dtype=np.float32)
    best_sequence = mean.copy()
    best_score = float("inf")
    start_state = np.asarray(start_state, dtype=np.float32)

    for _ in range(CEM_ITERATIONS):
        candidates = np.clip(
            rng.normal(mean, std, size=(CEM_CANDIDATES, horizon)), -1.0, 1.0
        ).astype(np.float32)
        states = np.repeat(start_state[None, :], CEM_CANDIDATES, axis=0)
        closest_distance = np.full(CEM_CANDIDATES, np.inf, dtype=np.float32)
        for step_index in range(horizon):
            states = model_step(states, candidates[:, step_index])
            position_error = states[:, 0] - target
            distance = position_error**2 + 0.05 * states[:, 1]**2
            closest_distance = np.minimum(closest_distance, distance)
        scores = closest_distance + 0.001 * np.mean(candidates**2, axis=1)
        elite_indices = np.argsort(scores)[:CEM_ELITES]
        elites = candidates[elite_indices]
        mean = elites.mean(axis=0)
        std = np.maximum(elites.std(axis=0), 0.08)
        candidate_index = int(np.argmin(scores))
        if float(scores[candidate_index]) < best_score:
            best_score = float(scores[candidate_index])
            best_sequence = candidates[candidate_index].copy()

    return best_sequence, best_score


def momentum_reference(start_state, goal):
    """Build a same-start reference path with the documented momentum rule."""
    state = np.asarray(start_state, dtype=np.float32).copy()
    states = [state.copy()]
    for _ in range(MAX_STEPS):
        action = 1.0 if state[1] >= 0.0 else -1.0
        state = model_step(state, action)
        states.append(state.copy())
        if state[0] >= goal and state[1] >= 0.0:
            break
    return np.asarray(states, dtype=np.float32)


def run_episode(env, goal, horizon, seed, mode):
    observation, _ = env.reset(seed=seed)
    observed_state = np.asarray(observation, dtype=np.float32)
    imagined_state = observed_state.copy()
    reference_path = (
        momentum_reference(observed_state, goal)
        if mode == "reference_subgoal_mpc" else None
    )
    planning_calls = 0
    rng_seed = seed + 100 * horizon + int(goal * 1000) + sum(map(ord, mode))
    rng = np.random.default_rng(rng_seed)

    for step_count in range(1, MAX_STEPS + 1):
        if mode == "momentum":
            action = 1.0 if observed_state[1] >= 0.0 else -1.0
        else:
            planning_state = imagined_state if mode == "pure_imagination" else observed_state
            if mode == "reference_subgoal_mpc":
                target_index = min(step_count + 4, len(reference_path) - 1)
                active_target = float(reference_path[target_index, 0])
            else:
                active_target = goal
            sequence, _ = cem_plan(planning_state, active_target, horizon, rng)
            action = float(sequence[0])
            planning_calls += 1

        next_observation, _, terminated, truncated, _ = env.step(
            np.asarray([action], dtype=np.float32)
        )
        observed_state = np.asarray(next_observation, dtype=np.float32)
        if mode == "pure_imagination":
            imagined_state = model_step(imagined_state, action)
        else:
            imagined_state = observed_state.copy()

        success = bool(observed_state[0] >= goal and observed_state[1] >= 0.0)
        if success or terminated or truncated:
            return {
                "status": "complete", "success": success, "steps": step_count,
                "planning_calls": planning_calls, "final_position": float(observed_state[0]),
                "final_velocity": float(observed_state[1]), "error": "",
            }

    return {
        "status": "complete", "success": False, "steps": MAX_STEPS,
        "planning_calls": planning_calls, "final_position": float(observed_state[0]),
        "final_velocity": float(observed_state[1]), "error": "",
    }


## Paired evaluation

Each method receives the same reset seed for a given goal and horizon. Subgoal MPC receives a target five steps ahead on a reference path generated from that same start by the momentum rule. This is a scripted reference, not an expert demonstration. The momentum baseline is evaluated once per goal because it does not use a rollout horizon. A missing or errored run stays in the planned denominator and is shown in the raw outcome string.


In [ ]:
def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (float("nan"), float("nan"))
    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = z * math.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return (max(0.0, center - margin), min(1.0, center + margin))


episode_records = []
summary_rows = []
env = gym.make("MountainCarContinuous-v0", max_episode_steps=MAX_STEPS)
evaluation_started = time.perf_counter()

for goal in GOALS:
    for horizon in HORIZONS:
        for mode in ("pure_imagination", "feedback_mpc", "reference_subgoal_mpc"):
            print(f"Running {mode}, goal={goal:.2f}, horizon={horizon}...")
            condition_records = []
            for seed in SEEDS:
                try:
                    result = run_episode(env, goal, horizon, seed, mode)
                    record = {
                        "mode": mode, "goal": goal, "horizon": horizon, "seed": seed, **result
                    }
                except Exception as error:
                    record = {
                        "mode": mode, "goal": goal, "horizon": horizon, "seed": seed,
                        "status": "error", "success": False, "steps": None,
                        "planning_calls": None, "final_position": None,
                        "final_velocity": None, "error": repr(error),
                    }
                condition_records.append(record)
                episode_records.append(record)
            completed = [row for row in condition_records if row["status"] == "complete"]
            successes = sum(bool(row["success"]) for row in condition_records)
            successful_steps = [row["steps"] for row in completed if row["success"]]
            calls = [row["planning_calls"] for row in completed]
            low, high = wilson_interval(successes, len(SEEDS))
            outcomes = "".join(
                "M" if row["status"] != "complete" else "S" if row["success"] else "F"
                for row in condition_records
            )
            summary_rows.append({
                "mode": mode, "goal": goal, "horizon": horizon,
                "planned": len(SEEDS), "completed": len(completed),
                "missing": len(SEEDS) - len(completed), "successes": successes,
                "success_rate": successes / len(SEEDS), "wilson_low": low,
                "wilson_high": high,
                "mean_success_steps": float(np.mean(successful_steps)) if successful_steps else float("nan"),
                "successful_episodes": len(successful_steps),
                "mean_planning_calls": float(np.mean(calls)) if calls else 0.0,
                "raw_outcomes": outcomes,
            })

    # The momentum rule has no rollout horizon, so run it once per goal.
    mode = "momentum"
    print(f"Running {mode}, goal={goal:.2f} (horizon-independent)...")
    condition_records = []
    for seed in SEEDS:
        try:
            result = run_episode(env, goal, HORIZONS[0], seed, mode)
            record = {"mode": mode, "goal": goal, "horizon": None, "seed": seed, **result}
        except Exception as error:
            record = {
                "mode": mode, "goal": goal, "horizon": None, "seed": seed,
                "status": "error", "success": False, "steps": None,
                "planning_calls": None, "final_position": None,
                "final_velocity": None, "error": repr(error),
            }
        condition_records.append(record)
        episode_records.append(record)
    completed = [row for row in condition_records if row["status"] == "complete"]
    successes = sum(bool(row["success"]) for row in condition_records)
    successful_steps = [row["steps"] for row in completed if row["success"]]
    low, high = wilson_interval(successes, len(SEEDS))
    outcomes = "".join(
        "M" if row["status"] != "complete" else "S" if row["success"] else "F"
        for row in condition_records
    )
    summary_rows.append({
        "mode": mode, "goal": goal, "horizon": None, "planned": len(SEEDS),
        "completed": len(completed), "missing": len(SEEDS) - len(completed),
        "successes": successes, "success_rate": successes / len(SEEDS),
        "wilson_low": low, "wilson_high": high,
        "mean_success_steps": float(np.mean(successful_steps)) if successful_steps else float("nan"),
        "successful_episodes": len(successful_steps), "mean_planning_calls": 0.0,
        "raw_outcomes": outcomes,
    })

env.close()
evaluation_seconds = time.perf_counter() - evaluation_started
print(f"Evaluation time: {evaluation_seconds:.1f} seconds")
print(f"Planned episodes: {len(summary_rows) * len(SEEDS)} condition episodes across {len(summary_rows)} conditions")

In [ ]:
def show_summary_table(rows):
    headers = ["Mode", "Goal", "Horizon", "Success", "Wilson 95%", "Steps (successful)", "Calls", "Done/missing"]
    body = []
    for row in rows:
        horizon = "n/a" if row["horizon"] is None else str(row["horizon"])
        steps = "n/a" if np.isnan(row["mean_success_steps"]) else f"{row['mean_success_steps']:.1f} ({row['successful_episodes']})"
        body.append([
            row["mode"], f"{row['goal']:.2f}", horizon,
            f"{row['successes']}/{row['planned']} ({100 * row['success_rate']:.1f}%)",
            f"[{100 * row['wilson_low']:.1f}, {100 * row['wilson_high']:.1f}]%",
            steps, f"{row['mean_planning_calls']:.1f}",
            f"{row['completed']}/{row['missing']}",
        ])
    table = "<table><thead><tr>" + "".join(f"<th>{item}</th>" for item in headers) + "</tr></thead><tbody>"
    for record in body:
        table += "<tr>" + "".join(f"<td>{item}</td>" for item in record) + "</tr>"
    table += "</tbody></table>"
    display(HTML(table))

show_summary_table(summary_rows)

In [ ]:
# Each character is the outcome for the seed at the same index in SEEDS:
# S = success, F = completed episode without success, M = missing/error.
raw_rows = [
    f"{row['mode']} | goal={row['goal']:.2f} | horizon={row['horizon'] or 'n/a'} | {row['raw_outcomes']}"
    for row in summary_rows
]
print("Seed order:", SEEDS)
print("\n".join(raw_rows))
errors = [row for row in episode_records if row["status"] != "complete"]
print(f"\nMissing/error runs: {len(errors)}")
for row in errors:
    print(row["mode"], row["goal"], row["horizon"], row["seed"], row["error"])

In [ ]:
planner_modes = ("pure_imagination", "feedback_mpc", "reference_subgoal_mpc")
colors = {
    "pure_imagination": "#386cb0",
    "feedback_mpc": "#f0027f",
    "reference_subgoal_mpc": "#7fc97f",
    "momentum": "#666666",
}
fig, axes = plt.subplots(1, len(GOALS), figsize=(14, 4), sharey=True)
for axis, goal in zip(axes, GOALS):
    for mode in planner_modes:
        rates = [
            next(row["success_rate"] for row in summary_rows
                 if row["mode"] == mode and row["goal"] == goal and row["horizon"] == horizon)
            for horizon in HORIZONS
        ]
        axis.plot(HORIZONS, rates, marker="o", label=mode.replace("_", " "), color=colors[mode])
    momentum_rate = next(
        row["success_rate"] for row in summary_rows
        if row["mode"] == "momentum" and row["goal"] == goal
    )
    axis.axhline(momentum_rate, color=colors["momentum"], linestyle="--", label="momentum (no horizon)")
    axis.set_title(f"Goal position {goal:.2f}")
    axis.set_xlabel("Rollout horizon (steps)")
    axis.set_xticks(HORIZONS)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.25)
axes[0].set_ylabel("Success rate (20 paired starts)")
axes[-1].legend(loc="lower right", fontsize=8)
fig.suptitle("Locally measured: MountainCar planning comparison")
fig.tight_layout()
plt.show()

## How to read the local result

Use the table and raw outcome strings to compare only these fixed seeds and settings. The Wilson intervals describe the 20 episode outcomes for one condition. They do not describe variation across trained models because this lesson trains no model.

The environment has exact deterministic state dynamics. If pure imagination and feedback MPC perform similarly, that is consistent with the model receiving no correction-worthy prediction error here. If a longer horizon or subgoal target helps, that is a result for this MountainCar setup and this planner score. Neither outcome confirms or refutes the paper.

The subgoal method receives the state five steps ahead on a reference path from a simple momentum controller. This is a teaching analogue of nearby demonstration subgoals. It gives the planner extra guidance, and its result must be read with that limit in mind. The paper uses expert trajectories from the same episode; this notebook does not.


## Sources

- Paper: [The Planning Limits of Latent World Models, arXiv:2609.39235v1](https://arxiv.org/html/2609.39235v1), especially Sections 6.1–6.3 and Appendix B.3.
- Environment dynamics and task definition: [Gymnasium continuous MountainCar implementation](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/continuous_mountain_car.py).
